In [ ]:
import pandas as pd
import glob
import os
import re
from pathlib import Path

CHASE_CREDIT_DIR     = '../original_csv/credit_cards/chase'
ALLIANT_CREDIT_DIR   = '../original_csv/credit_cards/alliant'
ALLIANT_CHECKING_DIR = '../original_csv/checking/alliant'
CHASE_CHECKING_DIR   = '../original_csv/checking/chase'

EXPENDITURES_CSV = '../expenditures.csv'
DEPOSITS_CSV     = '../deposits.csv'

DEDUP_COLS = ['date', 'description', 'amount']

def csv_files(directory):
    return (
        glob.glob(os.path.join(directory, '*.csv')) +
        glob.glob(os.path.join(directory, '*.CSV'))
    )

def parse_alliant_amount(val):
    val = str(val).strip()
    negative = val.startswith('(')
    val = re.sub(r'[\$,()\s]', '', val)
    return -float(val) if negative else float(val)

def load_existing(path):
    if Path(path).exists():
        df = pd.read_csv(path, index_col=0)
        df['date'] = pd.to_datetime(df['date'])
        return df
    return pd.DataFrame()

# --- Load existing output files ---
existing_exp = load_existing(EXPENDITURES_CSV)
existing_dep = load_existing(DEPOSITS_CSV)

# --- Parse source files ---

# Chase credit card
chase_credit_frames = []
for f in csv_files(CHASE_CREDIT_DIR):
    df = pd.read_csv(f)
    df = df[['Transaction Date', 'Description', 'Amount', 'Category']].copy()
    df.rename(columns={'Transaction Date': 'date', 'Description': 'description',
                       'Amount': 'amount', 'Category': 'category'}, inplace=True)
    df['account'] = 'chase_credit'
    chase_credit_frames.append(df)

chase_credit = pd.concat(chase_credit_frames, ignore_index=True) if chase_credit_frames else pd.DataFrame()
if not chase_credit.empty:
    chase_credit['date'] = pd.to_datetime(chase_credit['date'])
    # Chase: negative = expenditure, positive = deposit — already normalized

# Alliant credit card
alliant_credit_frames = []
for f in csv_files(ALLIANT_CREDIT_DIR):
    df = pd.read_csv(f)
    df = df[['Date', 'Description', 'Amount']].copy()
    df.rename(columns={'Date': 'date', 'Description': 'description'}, inplace=True)
    df['amount'] = df['Amount'].apply(parse_alliant_amount)
    df.drop(columns=['Amount'], inplace=True)
    df['category'] = ''
    df['account'] = 'alliant_credit'
    alliant_credit_frames.append(df)

alliant_credit = pd.concat(alliant_credit_frames, ignore_index=True) if alliant_credit_frames else pd.DataFrame()
if not alliant_credit.empty:
    alliant_credit['date'] = pd.to_datetime(alliant_credit['date'])
    # Alliant credit: positive = charge, parens = payment
    # Negate so negative = expenditure, positive = deposit (consistent with Chase)
    alliant_credit['amount'] = -alliant_credit['amount']

# Alliant checking
alliant_checking_frames = []
for f in csv_files(ALLIANT_CHECKING_DIR):
    df = pd.read_csv(f)
    df = df[['Date', 'Description', 'Amount']].copy()
    df.rename(columns={'Date': 'date', 'Description': 'description'}, inplace=True)
    df['amount'] = df['Amount'].apply(parse_alliant_amount)
    df.drop(columns=['Amount'], inplace=True)
    df['category'] = ''
    df['account'] = 'alliant_checking'
    alliant_checking_frames.append(df)

alliant_checking = pd.concat(alliant_checking_frames, ignore_index=True) if alliant_checking_frames else pd.DataFrame()
if not alliant_checking.empty:
    alliant_checking['date'] = pd.to_datetime(alliant_checking['date'])
    # Alliant checking: positive = deposit, parens/negative = withdrawal — already normalized

# Chase checking
chase_checking_frames = []
for f in csv_files(CHASE_CHECKING_DIR):
    df = pd.read_csv(f)
    df = df[['Transaction Date', 'Description', 'Amount']].copy()
    df.rename(columns={'Transaction Date': 'date', 'Description': 'description',
                       'Amount': 'amount'}, inplace=True)
    df['category'] = ''
    df['account'] = 'chase_checking'
    chase_checking_frames.append(df)

chase_checking = pd.concat(chase_checking_frames, ignore_index=True) if chase_checking_frames else pd.DataFrame()
if not chase_checking.empty:
    chase_checking['date'] = pd.to_datetime(chase_checking['date'])

# --- Combine all parsed data ---
all_sources = [df for df in [chase_credit, alliant_credit, alliant_checking, chase_checking] if not df.empty]
if not all_sources:
    print("No source files found.")
else:
    combined = pd.concat(all_sources, ignore_index=True)

    # --- Find new transactions not already in existing CSVs ---
    def find_new_rows(incoming, existing):
        """Return rows in incoming whose (date, description, amount) aren't in existing."""
        if existing.empty:
            return incoming
        existing_keys = existing[DEDUP_COLS].copy()
        existing_keys['date'] = pd.to_datetime(existing_keys['date'])
        merged = incoming.merge(existing_keys, on=DEDUP_COLS, how='left', indicator=True)
        return incoming[merged['_merge'] == 'left_only'].copy()

    raw_exp = combined[combined['amount'] < 0].copy()
    raw_dep = combined[combined['amount'] > 0].copy()

    new_exp = find_new_rows(raw_exp, existing_exp)
    new_dep = find_new_rows(raw_dep, existing_dep)

    # Add required columns to new expenditure rows
    new_exp['subcategory'] = pd.NA
    new_exp['merchant']    = pd.NA
    new_exp['ignore']      = False
    new_exp['necessity']   = pd.NA
    new_exp = new_exp[['date', 'description', 'amount', 'account', 'category',
                        'subcategory', 'merchant', 'ignore', 'necessity']]

    # Deposits only need category
    new_dep = new_dep[['date', 'description', 'amount', 'account', 'category']]

    # --- Merge new rows into existing and save ---
    expenditures = pd.concat([existing_exp, new_exp], ignore_index=True).sort_values('date')
    deposits     = pd.concat([existing_dep, new_dep], ignore_index=True).sort_values('date')

    expenditures.reset_index(drop=True, inplace=True)
    deposits.reset_index(drop=True, inplace=True)

    expenditures.to_csv(EXPENDITURES_CSV)
    deposits.to_csv(DEPOSITS_CSV)

    print(f"New expenditure rows added: {len(new_exp)}  (total: {len(expenditures)})")
    print(f"New deposit rows added:     {len(new_dep)}  (total: {len(deposits)})")